# Partial-return candidates among unmatched cancellations

This screen examines cancellations that the exact-quantity matcher did not match. It looks for eligible positive lines for the same customer and stock code dated strictly earlier. A larger earlier line is a possible partial-return candidate, not proof that the cancellation refers to that line. Recorded quantity is not verified demand.

The matcher selects cancellation lines with a `C` invoice, negative quantity, positive unit price, and a present customer ID. Positive comparison lines additionally require a non-`C` invoice, positive quantity and unit price, a present customer ID and invoice date, and a stock code matching the project's product-code regex.

Each unmatched cancellation belongs to exactly one of five groups: a larger earlier eligible purchase exists; no larger purchase exists but an equal-quantity earlier purchase does; only strictly smaller earlier purchases exist; no earlier purchase exists and the cancellation stock code matches the product-code regex; or no earlier purchase exists and the cancellation stock code fails the regex. Earlier lines must share customer and stock code and have a strictly earlier invoice date. Positive lines are candidate evidence, not proof of linked returns; recorded quantity is not verified demand.

In [1]:
from bisect import bisect_left
from hashlib import sha256
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root_candidates = (Path.cwd(), *Path.cwd().parents)
project_root = next(
    (candidate for candidate in project_root_candidates if (candidate / "src" / "clean.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root from the current directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.clean import PRODUCT_CODE, load_raw, match_cancelled_orders

workbook_path = project_root / "data" / "raw" / "Online Retail.xlsx"
workbook_hash_before = sha256(workbook_path.read_bytes()).hexdigest()
raw = load_raw(workbook_path)
raw_snapshot = raw.copy(deep=True)
matched_positive_index, matched_cancellation_index = match_cancelled_orders(raw)
matched_positive_index_set = set(matched_positive_index)

print(f"Workbook: {workbook_path.relative_to(project_root)}")
print(f"Raw rows: {len(raw):,}")
print(f"Matched cancellation lines: {len(matched_cancellation_index):,}")

Workbook: data\raw\Online Retail.xlsx
Raw rows: 541,909
Matched cancellation lines: 2,831


In [2]:
cancellation_mask = (
    raw["InvoiceNo"].astype(str).str.startswith("C")
    & raw["Quantity"].lt(0)
    & raw["UnitPrice"].gt(0)
    & raw["CustomerID"].notna()
)
cancellation_lines = raw.loc[cancellation_mask].copy()
cancellation_lines["product_code_matches"] = cancellation_lines["StockCode"].str.match(PRODUCT_CODE)
unmatched_cancellations = cancellation_lines.loc[
    ~cancellation_lines.index.isin(matched_cancellation_index)
].copy()

positive_mask = (
    ~raw["InvoiceNo"].astype(str).str.startswith("C")
    & raw["Quantity"].gt(0)
    & raw["CustomerID"].notna()
    & raw["InvoiceDate"].notna()
    & raw["UnitPrice"].gt(0)
    & raw["StockCode"].str.match(PRODUCT_CODE)
)
eligible_positive_lines = raw.loc[positive_mask]

positive_by_customer_product = {}
for key, lines in eligible_positive_lines.groupby(["CustomerID", "StockCode"], sort=False):
    ordered = lines.sort_values("InvoiceDate", kind="mergesort")
    positive_by_customer_product[key] = (
        ordered["InvoiceDate"].tolist(),
        ordered["Quantity"].tolist(),
        ordered.index.tolist(),
    )

group_larger = "larger earlier purchase exists"
group_equal = "no larger earlier purchase; equal-quantity earlier purchase exists"
group_smaller = "only strictly smaller earlier purchases exist"
group_none_valid_code = "no earlier purchase; product-code StockCode"
group_none_invalid_code = "no earlier purchase; StockCode fails PRODUCT_CODE"
group_order = [
    group_larger,
    group_equal,
    group_smaller,
    group_none_valid_code,
    group_none_invalid_code,
]
classified_rows = []

for cancellation in unmatched_cancellations.itertuples():
    returned_units = -cancellation.Quantity
    key = (cancellation.CustomerID, cancellation.StockCode)
    candidate_dates, candidate_quantities, candidate_indexes = positive_by_customer_product.get(
        key, ([], [], [])
    )
    earlier_stop = (
        bisect_left(candidate_dates, cancellation.InvoiceDate)
        if pd.notna(cancellation.InvoiceDate)
        else 0
    )
    earlier_quantities = candidate_quantities[:earlier_stop]
    earlier_indexes = candidate_indexes[:earlier_stop]
    equal_indexes = [
        index
        for index, quantity in zip(earlier_indexes, earlier_quantities)
        if quantity == returned_units
    ]
    matched_equal_count = sum(index in matched_positive_index_set for index in equal_indexes)

    if any(quantity > returned_units for quantity in earlier_quantities):
        group = group_larger
    elif equal_indexes:
        group = group_equal
    elif earlier_quantities:
        group = group_smaller
    elif cancellation.product_code_matches:
        group = group_none_valid_code
    else:
        group = group_none_invalid_code

    classified_rows.append(
        {
            "cancellation_index": cancellation.Index,
            "InvoiceNo": cancellation.InvoiceNo,
            "InvoiceDate": cancellation.InvoiceDate,
            "CustomerID": cancellation.CustomerID,
            "StockCode": cancellation.StockCode,
            "returned_units": returned_units,
            "earlier_purchase_count": len(earlier_quantities),
            "largest_earlier_quantity": max(earlier_quantities, default=None),
            "equal_sized_earlier_count": len(equal_indexes),
            "matched_equal_earlier_count": matched_equal_count,
            "group": group,
        }
    )

classified = pd.DataFrame(classified_rows)
if classified.empty:
    classified = pd.DataFrame(
        columns=[
            "cancellation_index", "InvoiceNo", "InvoiceDate", "CustomerID", "StockCode",
            "returned_units", "earlier_purchase_count", "largest_earlier_quantity",
            "equal_sized_earlier_count", "matched_equal_earlier_count", "group",
        ]
    )

print(f"Cancellation lines considered by matcher: {len(cancellation_lines):,}")
print(f"Unmatched cancellation lines screened: {len(unmatched_cancellations):,}")
print(f"Eligible positive lines: {len(eligible_positive_lines):,}")

Cancellation lines considered by matcher: 8,905
Unmatched cancellation lines screened: 6,074
Eligible positive lines: 396,337


In [3]:
summary_rows = []
for group in group_order:
    rows = classified.loc[classified["group"].eq(group)]
    summary_rows.append(
        {
            "group": group,
            "cancellation_count": len(rows),
            "returned_units": int(rows["returned_units"].sum()),
            "rows_with_equal_earlier_lines": int(rows["equal_sized_earlier_count"].gt(0).sum()),
            "all_equal_lines_matched": int(
                (
                    rows["equal_sized_earlier_count"].gt(0)
                    & rows["matched_equal_earlier_count"].eq(rows["equal_sized_earlier_count"])
                ).sum()
            ),
            "at_least_one_equal_line_unmatched": int(
                (
                    rows["equal_sized_earlier_count"].gt(rows["matched_equal_earlier_count"])
                ).sum()
            ),
        }
    )
summary = pd.DataFrame(summary_rows)
display(summary)

example_columns = [
    "InvoiceNo", "InvoiceDate", "CustomerID", "StockCode", "returned_units",
    "earlier_purchase_count", "largest_earlier_quantity", "equal_sized_earlier_count",
    "matched_equal_earlier_count",
]
for group in group_order:
    print(f"\nExamples: {group}")
    examples = classified.loc[classified["group"].eq(group), example_columns].head(5)
    display(examples.reset_index(drop=True))

def report_assertion(label, check):
    try:
        check()
    except AssertionError as error:
        print(f"FAIL: {label}: {error}")
    else:
        print(f"PASS: {label}")

group_equal_rows = classified.loc[classified["group"].eq(group_equal)].copy()
group_equal_rows["equal_lines_status"] = "at least one equal line unmatched"
group_equal_rows.loc[
    group_equal_rows["matched_equal_earlier_count"].eq(
        group_equal_rows["equal_sized_earlier_count"]
    ),
    "equal_lines_status",
] = "all equal lines already matched"
print("\nEqual-quantity group (b): matched-status counts")
display(
    group_equal_rows["equal_lines_status"]
    .value_counts()
    .reindex(["all equal lines already matched", "at least one equal line unmatched"], fill_value=0)
    .rename_axis("equal_lines_status")
    .to_frame("cancellation_count")
)
for status in ["all equal lines already matched", "at least one equal line unmatched"]:
    print(f"\nExamples: group (b), {status}")
    status_examples = group_equal_rows.loc[
        group_equal_rows["equal_lines_status"].eq(status), example_columns
    ].head(5)
    print(f"Available rows: {len(group_equal_rows.loc[group_equal_rows['equal_lines_status'].eq(status)]):,}; showing {len(status_examples):,} of 5 requested.")
    if status_examples.empty:
        print("No examples exist for this status in the screened data.")
    display(status_examples.reset_index(drop=True))

group_larger_equal_rows = classified.loc[
    classified["group"].eq(group_larger) & classified["equal_sized_earlier_count"].gt(0)
].copy()
group_larger_equal_rows["equal_lines_status"] = "at least one equal line unmatched"
group_larger_equal_rows.loc[
    group_larger_equal_rows["matched_equal_earlier_count"].eq(
        group_larger_equal_rows["equal_sized_earlier_count"]
    ),
    "equal_lines_status",
] = "all equal lines already matched"
print("\nGroup (a): cancellations that also have equal-quantity earlier lines")
print(f"Rows with an equal earlier line: {len(group_larger_equal_rows):,}")
display(group_larger_equal_rows["equal_lines_status"].value_counts().rename_axis("equal_lines_status").to_frame("cancellation_count"))
print("Examples: group (a) rows with equal-quantity earlier lines")
display(group_larger_equal_rows[example_columns].head(5).reset_index(drop=True))

def assert_groups_sum_to_expected():
    actual = int(summary["cancellation_count"].sum())
    assert actual == 6_074, f"five group counts sum to {actual:,}, expected 6,074"

def assert_every_unmatched_cancellation_classified_once():
    actual_indexes = classified["cancellation_index"]
    assert len(actual_indexes) == len(unmatched_cancellations), (
        f"classified {len(actual_indexes):,} rows for {len(unmatched_cancellations):,} unmatched cancellations"
    )
    assert actual_indexes.is_unique, "a cancellation was classified more than once"
    assert set(actual_indexes) == set(unmatched_cancellations.index), (
        "the classified indexes differ from unmatched cancellation indexes"
    )
    assert classified["group"].isin(group_order).all(), "an unmatched cancellation has no valid group"

def assert_raw_dataframe_unchanged():
    pd.testing.assert_frame_equal(raw, raw_snapshot)

def assert_workbook_unchanged():
    current_hash = sha256(workbook_path.read_bytes()).hexdigest()
    assert current_hash == workbook_hash_before, "workbook SHA-256 changed during this analysis"

print("\nValidation")
report_assertion("five group counts sum to 6,074", assert_groups_sum_to_expected)
report_assertion(
    "every unmatched cancellation is classified exactly once",
    assert_every_unmatched_cancellation_classified_once,
)
report_assertion("loaded raw DataFrame unchanged", assert_raw_dataframe_unchanged)
report_assertion("raw workbook unchanged", assert_workbook_unchanged)

,group,cancellation_count,returned_units,rows_with_equal_earlier_lines,all_equal_lines_matched,at_least_one_equal_line_unmatched
0,larger earlier purchase exists,4572,23914,16,16,0
1,no larger earlier purchase; equal-quantity ear...,21,131,21,21,0
2,only strictly smaller earlier purchases exist,84,1678,0,0,0
3,no earlier purchase; product-code StockCode,1031,18053,0,0,0
4,no earlier purchase; StockCode fails PRODUCT_CODE,366,5324,0,0,0



Examples: larger earlier purchase exists


,InvoiceNo,InvoiceDate,CustomerID,StockCode,returned_units,earlier_purchase_count,largest_earlier_quantity,equal_sized_earlier_count,matched_equal_earlier_count
0,C536506,2010-12-01 12:38:00,17897.0,22960,6,1,8.0,0,0
1,C536737,2010-12-02 13:05:00,15922.0,22798,2,1,8.0,0,0
2,C536758,2010-12-02 14:25:00,13767.0,21314,1,1,8.0,0,0
3,C536826,2010-12-02 17:30:00,17924.0,35004B,2,1,12.0,0,0
4,C536826,2010-12-02 17:30:00,17924.0,35004B,3,1,12.0,0,0



Examples: no larger earlier purchase; equal-quantity earlier purchase exists


,InvoiceNo,InvoiceDate,CustomerID,StockCode,returned_units,earlier_purchase_count,largest_earlier_quantity,equal_sized_earlier_count,matched_equal_earlier_count
0,C537406,2010-12-06 14:53:00,15502.0,22926,4,1,4.0,1,1
1,C537406,2010-12-06 14:53:00,15502.0,22927,4,1,4.0,1,1
2,C537406,2010-12-06 14:53:00,15502.0,22847,4,1,4.0,1,1
3,C537406,2010-12-06 14:53:00,15502.0,22849,4,1,4.0,1,1
4,C537408,2010-12-06 14:55:00,15502.0,47580,6,1,6.0,1,1


...

,cancellation_count
equal_lines_status,
all equal lines already matched,16


Examples: group (a) rows with equal-quantity earlier lines


,InvoiceNo,InvoiceDate,CustomerID,StockCode,returned_units,earlier_purchase_count,largest_earlier_quantity,equal_sized_earlier_count,matched_equal_earlier_count
0,C548355,2011-03-30 15:06:00,12921.0,22423,1,2,4.0,1,1
1,C549019,2011-04-05 15:22:00,12474.0,22236,1,2,2.0,1,1
2,C551659,2011-05-03 12:14:00,13668.0,85053,1,3,48.0,1,1
3,C553687,2011-05-18 14:01:00,15311.0,22646,1,3,72.0,1,1
4,C556643,2011-06-13 15:48:00,17841.0,23146,2,5,3.0,1,1



Validation
PASS: five group counts sum to 6,074
PASS: every unmatched cancellation is classified exactly once


PASS: loaded raw DataFrame unchanged
PASS: raw workbook unchanged
